# Telco Customer Churn Prediction — ICT6001 Week 01 Group Capstone

**Course:** ICT6001 — AI Programme (Week 1 Capstone)  
**Group:** Group 3 | **Members:** Joyce Yeo, KhoonSeng, Darren  
**Repo:** <To be updated>  
**Dataset:** Telco Customer Churn — Kaggle Playground Series S6E3 (594,194 rows × 21 features)  
**Source:** https://www.kaggle.com/competitions/playground-series-s6e3/data  
**Target:** `Churn` (binary — 1 = churned, 0 = retained)

---

### Notebook Roadmap

| Section | Part | CRISP-DM Phase | Description |
|---------|------|----------------|-------------|
| §1 | A | Data Preparation | Advanced Data Prep & Pipeline Engineering |
| §2 | B | Modelling | Champion Model Selection via 5-fold CV |
| §3 | C | Modelling | Hyperparameter Tuning |
| §4 | E | Deployment | Business Decision Making |

## Set-up Environment

In [ ]:
%matplotlib inline
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, GridSearchCV, cross_val_predict
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectPercentile, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, recall_score,
    balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="Set2")
print("Environment ready.")

In [ ]:
# Portable data load — works locally and in CI/headless runs.
# Set CAPSTONE_SAMPLE=N for fast iteration (~50k); 0 = full 594k.
DATA_DIR = os.environ.get("CAPSTONE_DATA", "data")
df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))

_n = int(os.environ.get("CAPSTONE_SAMPLE", "0"))
if _n:
    df = df.groupby("Churn", group_keys=False).sample(
        frac=min(1.0, _n / len(df)), random_state=RANDOM_STATE
    )
    print(f"[DEV MODE] stratified sample: {len(df):,} rows")
else:
    print(f"[FULL DATA] {len(df):,} rows")

df.shape

In [ ]:
# Churn → 0/1; coerce TotalCharges blanks to NaN (guard for test-set edge cases).
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

X = df.drop(columns=["id", "Churn"])
y = df["Churn"]

# 80 / 20 stratified split.  X_test is locked until §4 (touched exactly once).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"train : {X_train.shape}   test : {X_test.shape}")
print(f"Churn rate — train: {y_train.mean():.3f}   test: {y_test.mean():.3f}")

---
## 1. Advanced Data Preparation & Pipeline Engineering — Part A
**CRISP-DM Phase:** Data Preparation

**Purpose:** Finalise feature engineering and build a leakage-safe preprocessing pipeline.
All transforms (imputation, encoding, scaling, SMOTE oversampling) are encapsulated inside
a formal `sklearn.pipeline.Pipeline` so that no information from validation folds leaks
into the training distribution.


### 1.1 Exploratory Data Analysis

**Purpose:** Surface key patterns that inform feature decisions downstream.

In [ ]:
# Class distribution
churn_counts = y.value_counts().sort_index()
pct = (churn_counts / len(y) * 100).round(1)
print(churn_counts.rename({0: "No Churn", 1: "Churn"}).to_string())
print(f"\nChurn rate: {pct[1]:.1f}%  → moderate class imbalance; SMOTE addresses this in §1.3")

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(["No Churn (0)", "Churn (1)"], churn_counts.values,
              color=["#27ae60", "#e74c3c"], edgecolor="white")
for bar, val in zip(bars, churn_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
            f"{val:,}\n({val/len(y)*100:.1f}%)", ha="center", va="bottom", fontsize=9)
ax.set_title("Churn Distribution", fontweight="bold")
ax.set_ylabel("Count")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Tenure distribution
for label, color in [(0, "#27ae60"), (1, "#e74c3c")]:
    subset = df[df["Churn"] == label]["tenure"]
    axes[0].hist(subset, bins=30, alpha=0.55, color=color,
                 label=f"Churn={label}", density=True)
axes[0].set_title("Tenure vs Churn", fontweight="bold")
axes[0].set_xlabel("Tenure (months)"); axes[0].legend()

# 2. MonthlyCharges boxplot
df_plot = df.copy(); df_plot["Churn"] = df_plot["Churn"].map({0: "No", 1: "Yes"})
df_plot.boxplot(column="MonthlyCharges", by="Churn", ax=axes[1],
                medianprops=dict(color="#e74c3c", linewidth=2))
axes[1].set_title("MonthlyCharges vs Churn", fontweight="bold")
axes[1].set_xlabel("Churn"); plt.sca(axes[1]); plt.title("")

# 3. Contract type
contract_churn = df.groupby(["Contract", "Churn"]).size().unstack(fill_value=0)
contract_churn.plot(kind="bar", ax=axes[2],
                    color=["#27ae60", "#e74c3c"], edgecolor="white", rot=30)
axes[2].set_title("Contract Type vs Churn", fontweight="bold")
axes[2].set_xlabel(""); axes[2].legend(["No Churn", "Churn"])

plt.suptitle("Key Drivers of Churn (EDA)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()

print('''
EDA Insights
  1. Tenure: New customers (<12 months) have highest churn risk.
             Loyalty builds over time -- retention focus on the first year.
  2. MonthlyCharges: Churners pay ~$15-20 more per month (pricing sensitivity).
  3. Contract type: Month-to-month customers churn at ~42%;
                    two-year contract customers at <3%. Upgrading contracts is
                    the strongest single intervention lever.
''')

### 1.2 Missing Values & Data Quality

**Purpose:** Confirm completeness. `TotalCharges` was coerced to numeric above to guard against blank-string entries common in the raw telco dataset.

In [ ]:
missing = df.isnull().sum()
tc_nulls = missing["TotalCharges"]
other_nulls = missing.drop("TotalCharges").sum()
print(f"TotalCharges NaNs after coercion : {tc_nulls}")
print(f"All other columns missing        : {other_nulls}")
print("\nConclusion: no structural missing data.")

### 1.3 Feature Typing & Leakage-Safe Pipeline

**Purpose:** Encapsulate all transforms in a `ColumnTransformer → ImbPipeline`.
SMOTE is placed *inside* the pipeline — oversampling occurs only on training folds;
validation folds are never oversampled.

**Pipeline architecture:**
```
ImbPipeline
├── preprocessor (ColumnTransformer)
│   ├── num_pipe : StandardScaler
│   └── cat_pipe : OneHotEncoder(handle_unknown=ignore)
│                  → SelectPercentile(chi2, percentile=50)
├── smote       : SMOTE(random_state=42)   ← touches training fold only
└── classifier  : <model>
```

`SelectPercentile(chi2, 50)` retains the top-50% of one-hot columns by χ² association
with the target — reducing dimensionality and discarding low-signal binary dummies.

In [ ]:
num_features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
cat_features = [c for c in X.columns if c not in num_features]

print(f"Numeric     ({len(num_features)}): {num_features}")
print(f"Categorical ({len(cat_features)}): {cat_features}")

num_pipe = Pipeline([
    ("scaler",  StandardScaler()),
])
cat_pipe = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ("selector", SelectPercentile(score_func=chi2, percentile=50)),
])
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", cat_pipe, cat_features),
], remainder="drop")

print("\nPreprocessor built. SMOTE will be added per-model in §2.")

---
## 2. Champion Model Selection — Part B
**CRISP-DM Phase:** Modelling

### 2.1 Primary Evaluation Metric

**Why ROC-AUC?**
- The dataset is imbalanced (~22.5% churn). Accuracy rewards the majority class.
- ROC-AUC measures a model's ability to **rank** churners above non-churners across
  *all* thresholds — threshold-independent, so it is a fair between-family comparison.
- It directly supports §4's threshold-shift analysis: higher AUC = more headroom
  to tune the threshold for the business's specific FN cost structure.
- **Recall** is tracked as a secondary metric: it captures how many real churners the
  model catches, directly reflecting the business cost of a missed churner.
  Threshold-dependent metrics (F1, precision) are intentionally omitted here;
  the operating point is fixed by the recall-driven threshold selection in §4.

### 2.2 Candidate Model Families

Three architecturally distinct families compared at representative defaults:

| Family | Model | Rationale |
|--------|-------|-----------|
| Linear | Logistic Regression | Baseline; interpretable; linear decision boundary |
| Bagging | Random Forest | Variance reduction; built-in feature importance |
| Boosting | XGBoost | Sequentially corrects errors; strong on tabular data |

In [ ]:
skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = ["roc_auc", "recall"]

# Representative defaults — max_iter=1000 prevents ConvergenceWarning for LR
# (numerical setting, not a tuned hyperparameter). Tuning follows in §3.
candidates = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":             XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE,
                                          verbosity=0, n_jobs=-1),
}

cv_results = {}
for name, clf in candidates.items():
    pipe = ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote",        SMOTE(random_state=RANDOM_STATE)),
        ("classifier",   clf),
    ])
    scores = cross_validate(pipe, X_train, y_train,
                            cv=skf, scoring=SCORING, n_jobs=-1)
    cv_results[name] = {m: scores[f"test_{m}"] for m in SCORING}
    auc = scores["test_roc_auc"]
    print(f"{name:22}  ROC-AUC {auc.mean():.4f} ± {auc.std():.4f}")

print("\n5-fold CV complete.")

In [ ]:
# Results table
rows = []
for name, res in cv_results.items():
    row = {"Model": name}
    for m in SCORING:
        row[f"{m} mean"] = res[m].mean()
        row[f"{m} std"]  = res[m].std()
    rows.append(row)
cv_df = pd.DataFrame(rows).set_index("Model")

print("CV Results — mean ± std across 5 folds\n")
mean_cols = [c for c in cv_df.columns if "mean" in c]
for name, row in cv_df.iterrows():
    parts = "  |  ".join(
        f"{m.replace(' mean',''):>12} {row[m]:.4f}±{row[m.replace('mean','std')]:.4f}"
        for m in mean_cols
    )
    print(f"  {name:<22}  {parts}")

In [ ]:
models = list(cv_results.keys())
x = np.arange(len(models))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
b1 = ax.bar(x - w/2, [cv_results[m]["roc_auc"].mean() for m in models], w,
            yerr=[cv_results[m]["roc_auc"].std() for m in models],
            label="ROC-AUC", color="#2980b9", capsize=4, edgecolor="white")
b2 = ax.bar(x + w/2, [cv_results[m]["recall"].mean() for m in models], w,
            yerr=[cv_results[m]["recall"].std() for m in models],
            label="Recall", color="#e67e22", capsize=4, edgecolor="white")
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
            f"{h:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(models)
ax.set_ylim(0.50, 1.0); ax.set_ylabel("Score"); ax.legend()
ax.set_title("5-fold CV: ROC-AUC and Recall by Model Family", fontweight="bold")
plt.tight_layout(); plt.show()

### 2.3 Key Findings & Champion Declaration

The three families were screened at representative **default** hyperparameters using 5-fold CV
ROC-AUC (§2.2). The champion is the family with the **highest default-config CV ROC-AUC**.
Only this champion is then carried into hyperparameter tuning in §3 — tuning all three would
exceed the ≤50-iteration budget for marginal gain.

*A single champion is declared data-driven from the CV results above.
The winning configuration is refit on the full training set in §3.*

In [ ]:
# Data-driven champion: highest mean CV ROC-AUC at default params
CHAMPION_NAME = max(cv_results, key=lambda m: cv_results[m]["roc_auc"].mean())
print(f"Champion: {CHAMPION_NAME}  "
      f"(ROC-AUC {cv_results[CHAMPION_NAME]['roc_auc'].mean():.4f} "
      f"± {cv_results[CHAMPION_NAME]['roc_auc'].std():.4f})")
print("Hyperparameter tuning and refit on the champion follows in §3.")

---
## 3. Hyperparameter Tuning — Part C
**CRISP-DM Phase:** Modelling (refinement)

**Purpose:** Apply GridSearchCV to the champion family to find its optimal hyperparameter
configuration, then refit on the full training set.

### 3.1 Tuning Strategy

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Search method | GridSearchCV | Exhaustive over small grids; reproducible |
| Scope | Champion model only | One champion declared in §2; tune it alone |
| Iterations cap | ≤50 *candidate configs* | Champion grid ≤12 configs (well within cap) |
| CV folds | 5-fold stratified | Validation repeats, not search iterations |
| Tune dataset | 100k stratified subsample of X_train | 4–5× faster; winning config refit on full 475k |
| refit metric | `roc_auc` | Threshold-independent ranking; recall optimised in §4 |
| n_jobs | 2 | SMOTE inside pipeline; `-1` causes memory blow-up |

### 3.2 Tuning Log

In [ ]:
# Champion grids — only the champion's entry is used
_champion_grids = {
    "Logistic Regression": {
        "classifier__C":       [0.01, 0.1, 1, 10],
        "classifier__penalty": ["l1", "l2"],
    },
    "Random Forest": {
        "classifier__n_estimators": [200, 400],
        "classifier__max_depth":    [10, 20, None],
        "classifier__max_features": ["sqrt", "log2"],
    },
    "XGBoost": {
        "classifier__max_depth":     [3, 5, 7],
        "classifier__learning_rate": [0.05, 0.1],
        "classifier__n_estimators":  [200, 400],
    },
}
_champion_clfs = {
    "Logistic Regression": LogisticRegression(max_iter=1000, solver="saga",
                                               random_state=RANDOM_STATE),
    "Random Forest":       RandomForestClassifier(n_jobs=1, random_state=RANDOM_STATE),
    "XGBoost":             XGBClassifier(eval_metric="logloss", verbosity=0,
                                          n_jobs=1, random_state=RANDOM_STATE),
}

# ── Tuning subsample ──────────────────────────────────────────────────────
TUNE_SIZE = 100_000
if len(X_train) > TUNE_SIZE:
    X_tune, _, y_tune, _ = train_test_split(
        X_train, y_train,
        train_size=TUNE_SIZE, stratify=y_train, random_state=RANDOM_STATE
    )
else:
    X_tune, y_tune = X_train, y_train   # guard for --sample dev runs

n_configs = 1
for v in _champion_grids[CHAMPION_NAME].values():
    n_configs *= len(v)
print(f"Champion      : {CHAMPION_NAME}")
print(f"Tuning subset : {X_tune.shape}  (churn rate {y_tune.mean():.3f})")
print(f"Grid          : {n_configs} candidate configs × 5-fold CV")

# ── GridSearchCV ──────────────────────────────────────────────────────────
champion_search = GridSearchCV(
    estimator=ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote",        SMOTE(random_state=RANDOM_STATE)),
        ("classifier",   _champion_clfs[CHAMPION_NAME]),
    ]),
    param_grid=_champion_grids[CHAMPION_NAME],
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring={"roc_auc": "roc_auc", "recall": "recall"},
    refit="roc_auc",
    n_jobs=2,
    verbose=1,
)
champion_search.fit(X_tune, y_tune)

In [ ]:
idx = champion_search.best_index_
r   = champion_search.cv_results_

print("╔══════════════════════════════════════════════════════════════════════════════╗")
print("║                         HYPERPARAMETER TUNING LOG                           ║")
print("╚══════════════════════════════════════════════════════════════════════════════╝\n")
print(f"  {CHAMPION_NAME}")
for k, v in champion_search.best_params_.items():
    label = k.replace("classifier__", "")
    print(f"    {label:<20}: {v}")
print(f"    {'CV ROC-AUC':<20}: {r['mean_test_roc_auc'][idx]:.4f} ± {r['std_test_roc_auc'][idx]:.4f}")
print(f"    {'CV Recall':<20}: {r['mean_test_recall'][idx]:.4f} ± {r['std_test_recall'][idx]:.4f}")

print(f"\nRefitting on full X_train ({X_train.shape[0]:,} rows)…")
champion_pipe = clone(champion_search.best_estimator_)
champion_pipe.fit(X_train, y_train)
print("Refit complete. Champion pipeline locked for §4 evaluation.")

In [ ]:
print("Running cross_val_predict for validation probabilities (train only)…")
val_proba = cross_val_predict(
    champion_pipe, X_train, y_train,
    cv=skf, method="predict_proba", n_jobs=2
)[:, 1]
print(f"Done. {len(val_proba):,} validation predictions.")

---
## 4. Business Decision Making — Part E
**CRISP-DM Phase:** Deployment

### 4.1 Business Context

**Problem restatement:** Predict which customers will churn so the retention team can
intervene *before* the customer leaves.

**Asymmetric error costs:**

| Error | Consequence | Severity |
|-------|------------|----------|
| **False Negative** — miss a churner | Customer leaves uncontacted → lost lifetime value | **CRITICAL** |
| **False Positive** — flag a loyal customer | Wasted retention offer (a comparatively cheap, recoverable cost) | Moderate |

**Operating-point direction:** Missing a churner (FN) is far costlier than a wasted retention
offer (FP), so we prioritise **recall**. We require recall ≥ 0.80 and, among all thresholds
meeting that floor, select the **highest** one to keep false positives in check. Because SMOTE
shifts predicted probabilities upward, this operating point sits **above** the naïve 0.5 cut
(≈0.61), not below it.

*Per brief guidelines, we argue the direction and a justifiable operating point —
not the exact mathematical optimum.*

In [ ]:
# Recall vs threshold on validation-fold probabilities (no hold-out access)
thresholds = np.arange(0.15, 0.75, 0.01)
records = []
for t in thresholds:
    preds = (val_proba >= t).astype(int)
    records.append({
        "threshold": round(t, 2),
        "recall":    recall_score(y_train, preds, zero_division=0),
    })
thresh_df = pd.DataFrame(records)

# Select tightest threshold that still achieves recall >= 0.80
valid = thresh_df[thresh_df["recall"] >= 0.80]
if len(valid):
    THRESHOLD = float(valid["threshold"].max())
else:
    THRESHOLD = float(thresh_df.loc[thresh_df["recall"].idxmax(), "threshold"])

THRESHOLD = round(THRESHOLD, 2)
sel_recall = thresh_df.loc[thresh_df["threshold"] == THRESHOLD, "recall"].values[0]
print(f"Selected threshold : {THRESHOLD}")
print(f"  Recall    : {sel_recall:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresh_df["threshold"], thresh_df["recall"], label="Recall", color="#e74c3c", lw=2)

ax.axvline(0.50,      color="gray",    linestyle="--", alpha=0.7, lw=1.5, label="Default 0.50")
ax.axvline(THRESHOLD, color="#e74c3c", linestyle="-",  lw=2,
           label=f"Selected {THRESHOLD} (recall ≥ 0.80)")

ax.set_xlabel("Classification Threshold", fontsize=11)
ax.set_ylabel("Recall", fontsize=11)
ax.set_title("Recall vs Threshold", fontweight="bold", fontsize=12)
ax.legend(fontsize=9); ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

print(f"""
Justification (threshold = {THRESHOLD}):
  - Missing a churner (FN) costs far more than a wasted retention offer (FP),
    so we prioritise recall and require recall >= 0.80.
  - Among all thresholds meeting that floor, {THRESHOLD} is the highest -- it holds
    >=80% recall while keeping false positives as low as possible.
  - SMOTE inflates predicted probabilities, so this operating point sits above 0.5, not below.
  - Exact optimum left to stakeholder negotiation; {THRESHOLD} is a justifiable operating point.
""")

### 4.2 Final Hold-Out Evaluation

⚠️ **The hold-out set is accessed exactly once — here.**
Champion is fit on full `train`; locked threshold is applied; metrics reported.

In [ ]:
print("Fitting champion on full train…")
champion_pipe.fit(X_train, y_train)

# ── HOLD-OUT EVALUATED ONCE ──────────────────────────────────────────────────
test_proba = champion_pipe.predict_proba(X_test)[:, 1]
test_pred  = (test_proba >= THRESHOLD).astype(int)

hold_auc  = roc_auc_score(y_test, test_proba)
hold_rec  = recall_score(y_test, test_pred)
hold_bacc = balanced_accuracy_score(y_test, test_pred)

print(f"\n══════════════════════════════════════════════════════")
print(f"  FINAL HOLD-OUT DEPLOYMENT METRICS  (threshold={THRESHOLD})")
print(f"══════════════════════════════════════════════════════")
print(f"  ROC-AUC           : {hold_auc:.4f}")
print(f"  Recall            : {hold_rec:.4f}")
print(f"  Balanced Accuracy : {hold_bacc:.4f}")
print(f"══════════════════════════════════════════════════════")

fig, ax = plt.subplots(figsize=(4, 4))
cm = confusion_matrix(y_test, test_pred)
ConfusionMatrixDisplay(cm, display_labels=["No Churn", "Churn"]).plot(
    ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Hold-Out Confusion Matrix\n(threshold={THRESHOLD})", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
_family_desc = {
    "Logistic Regression": "linear (logistic regression)",
    "Random Forest":       "bagging (random forest)",
    "XGBoost":             "boosting (XGBoost)",
}.get(CHAMPION_NAME, CHAMPION_NAME)

print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║                    EXECUTIVE BUSINESS SUMMARY                        ║
╚══════════════════════════════════════════════════════════════════════╝

Problem
  Telco churn erodes lifetime customer value. With ~22.5% of customers churning,
  proactive retention intervention is critical and cost-effective.

Solution
  {CHAMPION_NAME} classifier ({_family_desc}) in a leakage-safe ImbPipeline with
  SMOTE oversampling. Champion selected by 5-fold CV ROC-AUC at default hyperparameters across
  Logistic Regression, Random Forest, and XGBoost; the leader ({CHAMPION_NAME}) was
  then hyperparameter-tuned in §3.

Hold-Out Deployment Performance  [threshold = {THRESHOLD}]
  · ROC-AUC {hold_auc:.3f} — the model correctly ranks churners above non-churners
    {hold_auc*100:.1f}% of the time across all possible thresholds.
  · Recall  {hold_rec:.3f} — catches {hold_rec*100:.1f}% of actual churners before they leave.

Business Decision
  Operating threshold set to {THRESHOLD} (recall-first: highest cut still holding >=80% recall).
  Rationale: missing a churner costs far more than a wasted retention offer, so we prioritise
  recall; SMOTE shifts predicted probabilities upward, so this point sits above 0.5.
  At this operating point the retention team captures ~{hold_rec*100:.0f}% of churners.
  Recommendation: tier retention offers by predicted probability (high/medium/low),
  concentrating premium spend on the highest-probability churners.
""")